In [1]:
import pandas as pd
from pathlib import Path
from warnings import filterwarnings

filterwarnings("ignore")

In [3]:
PASTA_ATUAL = Path.cwd()

CAMINHO = (Path.cwd().parent/ "data"/ "raw"/ "Preços semestrais - AUTOMOTIVOS_2026.01.csv")

data_anp = pd.read_csv(
    CAMINHO,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",  
    dtype={"CNPJ da Revenda": "string", "Cep": "string", "Numero Rua": "string"},
)

data_anp["Data da Coleta"] = pd.to_datetime(data_anp["Data da Coleta"], format="%d/%m/%Y")

In [4]:
data_anp.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
0,N,AC,CRUZEIRO DO SUL,AMAZONIA COMERCIO DE DERIVADOS DE PETROLEO LTDA,01.492.748/0003-83,AVENIDA COPACABANA,440,NaN,COPACABANA,69980-000,GASOLINA,2026-01-02,7.97,NaN,R$ / litro,IPIRANGA
1,N,AC,CRUZEIRO DO SUL,AMAZONIA COMERCIO DE DERIVADOS DE PETROLEO LTDA,01.492.748/0003-83,AVENIDA COPACABANA,440,NaN,COPACABANA,69980-000,DIESEL,2026-01-02,8.15,NaN,R$ / litro,IPIRANGA
2,N,AC,CRUZEIRO DO SUL,AMAZONIA COMERCIO DE DERIVADOS DE PETROLEO LTDA,01.492.748/0003-83,AVENIDA COPACABANA,440,NaN,COPACABANA,69980-000,DIESEL S10,2026-01-02,8.17,NaN,R$ / litro,IPIRANGA
3,N,AC,RIO BRANCO,AUTO POSTO AMAPA LTDA,00.529.581/0001-53,VIA CHICO MENDES,3570,NaN,AREAL,69906-119,GASOLINA,2026-01-02,7.29,NaN,R$ / litro,VIBRA
4,N,AC,RIO BRANCO,AUTO POSTO AMAPA LTDA,00.529.581/0001-53,VIA CHICO MENDES,3570,NaN,AREAL,69906-119,GASOLINA ADITIVADA,2026-01-02,7.39,NaN,R$ / litro,VIBRA


In [5]:
data_anp["Data da Coleta"].dt.strftime("%d/%m/%Y").agg(["min", "max"])


min    01/01/2026
max    31/03/2026
Name: Data da Coleta, dtype: str

In [ ]:
data_anp.shape


(422418, 16)

In [6]:
data_anp.info()

<class 'pandas.DataFrame'>
RangeIndex: 422418 entries, 0 to 422417
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   Regiao - Sigla     422418 non-null  str           
 1   Estado - Sigla     422418 non-null  str           
 2   Municipio          422418 non-null  str           
 3   Revenda            422418 non-null  str           
 4   CNPJ da Revenda    422418 non-null  string        
 5   Nome da Rua        422418 non-null  str           
 6   Numero Rua         422412 non-null  string        
 7   Complemento        96159 non-null   str           
 8   Bairro             421690 non-null  str           
 9   Cep                422418 non-null  string        
 10  Produto            422418 non-null  str           
 11  Data da Coleta     422418 non-null  datetime64[us]
 12  Valor de Venda     422418 non-null  float64       
 13  Valor de Compra    0 non-null       float64       
 14 

In [24]:
data_anp.isna().sum()

Regiao - Sigla            0
Estado - Sigla            0
Municipio                 0
Revenda                   0
CNPJ da Revenda           0
Nome da Rua               0
Numero Rua                6
Complemento          326259
Bairro                  728
Cep                       0
Produto                   0
Data da Coleta            0
Valor de Venda            0
Valor de Compra      422418
Unidade de Medida         0
Bandeira                  0
dtype: int64

In [12]:
data_anp.query("Bairro.isna()").head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
316,SE,SP,JOSE BONIFACIO,AUTO POSTO EDUARDO LIMITADA,46.603.957/0001-23,AVENIDA 9 DE JULHO,284,NaN,NaN,15200-000,GASOLINA,2026-01-02,6.09,NaN,R$ / litro,RAIZEN
317,SE,SP,JOSE BONIFACIO,AUTO POSTO EDUARDO LIMITADA,46.603.957/0001-23,AVENIDA 9 DE JULHO,284,NaN,NaN,15200-000,GASOLINA ADITIVADA,2026-01-02,6.39,NaN,R$ / litro,RAIZEN
318,SE,SP,JOSE BONIFACIO,AUTO POSTO EDUARDO LIMITADA,46.603.957/0001-23,AVENIDA 9 DE JULHO,284,NaN,NaN,15200-000,DIESEL,2026-01-02,5.89,NaN,R$ / litro,RAIZEN
319,SE,SP,JOSE BONIFACIO,AUTO POSTO EDUARDO LIMITADA,46.603.957/0001-23,AVENIDA 9 DE JULHO,284,NaN,NaN,15200-000,ETANOL,2026-01-02,4.29,NaN,R$ / litro,RAIZEN
2203,SE,MG,CARATINGA,J MENDES DE AQUINO & CIA. LTDA,42.978.668/0001-58,PRACA CESARIO ALVIM,80,NaN,NaN,35300-036,GASOLINA,2026-01-06,6.29,NaN,R$ / litro,BRANCA


### Enriquecer bairros ausentes pelo CEP

A consulta usa o ViaCEP apenas para as linhas sem bairro e consulta cada CEP distinto uma vez. O resultado fica em `Bairro consultado`; a coluna original `Bairro` não foi alterada. CEPs sem retorno continuam sem preenchimento.

In [13]:
import json
import re
import time
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

# Considera ausentes tanto valores nulos quanto textos vazios.
bairro_vazio = data_anp["Bairro"].isna() | data_anp["Bairro"].astype("string").str.strip().eq("")

def limpar_cep(valor):
    if pd.isna(valor):
        return None
    cep = re.sub(r"\D", "", str(valor))
    return cep if len(cep) == 8 else None

# Consulta apenas CEPs distintos das linhas sem bairro.
ceps = (
    data_anp.loc[bairro_vazio, "Cep"]
    .map(limpar_cep)
    .dropna()
    .drop_duplicates()
)

consultas_cep = {}
for cep in ceps:
    try:
        url = f"https://viacep.com.br/ws/{cep}/json/"
        with urlopen(url, timeout=10) as resposta:
            endereco = json.loads(resposta.read().decode("utf-8"))
        consultas_cep[cep] = endereco.get("bairro") if not endereco.get("erro") else None
    except (HTTPError, URLError, TimeoutError, json.JSONDecodeError) as erro:
        consultas_cep[cep] = None
        print(f"Falha ao consultar o CEP {cep}: {erro}")
    time.sleep(0.1)  # pequena pausa entre consultas

# Mant?m a informa??o consultada separada da coluna original da ANP.
data_anp["Bairro consultado"] = pd.NA
ceps_linhas_vazias = data_anp.loc[bairro_vazio, "Cep"].map(limpar_cep)
data_anp.loc[bairro_vazio, "Bairro consultado"] = ceps_linhas_vazias.map(consultas_cep)

print(f"Linhas sem bairro original: {bairro_vazio.sum()}")
print(f"CEPs distintos consultados: {len(ceps)}")
print(f"Linhas que receberam bairro consultado: {data_anp.loc[bairro_vazio, 'Bairro consultado'].notna().sum()}")
data_anp.loc[bairro_vazio, ["Municipio", "Nome da Rua", "Cep", "Bairro", "Bairro consultado"]].head(10)

Linhas sem bairro original: 728
CEPs distintos consultados: 12
Linhas que receberam bairro consultado: 468


,Municipio,Nome da Rua,Cep,Bairro,Bairro consultado
316,JOSE BONIFACIO,AVENIDA 9 DE JULHO,15200-000,NaN,NaN
317,JOSE BONIFACIO,AVENIDA 9 DE JULHO,15200-000,NaN,NaN
318,JOSE BONIFACIO,AVENIDA 9 DE JULHO,15200-000,NaN,NaN
319,JOSE BONIFACIO,AVENIDA 9 DE JULHO,15200-000,NaN,NaN
2203,CARATINGA,PRACA CESARIO ALVIM,35300-036,NaN,Centro
2204,CARATINGA,PRACA CESARIO ALVIM,35300-036,NaN,Centro
2205,CARATINGA,PRACA CESARIO ALVIM,35300-036,NaN,Centro
3084,CASTANHAL,AVENIDA PRESIDENTE VARGAS,68742-005,NaN,Nova Olinda
3085,CASTANHAL,AVENIDA PRESIDENTE VARGAS,68742-005,NaN,Nova Olinda
3521,TERESINA,AVENIDA MIGUEL ROSA,64001-973,NaN,Centro


In [14]:
data_anp_at = data_anp.copy()

data_anp_at["Bairro"] = (
    data_anp_at["Bairro"]
    .fillna(data_anp_at["Bairro consultado"])
)

In [20]:
data_anp_at.isna().sum()

Regiao - Sigla         0
Estado - Sigla         0
Municipio              0
Revenda                0
CNPJ da Revenda        0
Nome da Rua            0
Numero Rua             6
Bairro               260
Cep                    0
Produto                0
Data da Coleta         0
Valor de Venda         0
Unidade de Medida      0
Bandeira               0
dtype: int64

In [ ]:
data_anp_at.drop(columns=["Bairro consultado"], inplace=True)
data_anp_at.drop(columns=["Complemento"], inplace=True)
data_anp_at.drop(columns=["Valor de Compra"], inplace=True)

In [21]:
data_anp_at.head()

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Unidade de Medida,Bandeira
0,N,AC,CRUZEIRO DO SUL,AMAZONIA COMERCIO DE DERIVADOS DE PETROLEO LTDA,01.492.748/0003-83,AVENIDA COPACABANA,440,COPACABANA,69980-000,GASOLINA,2026-01-02,7.97,R$ / litro,IPIRANGA
1,N,AC,CRUZEIRO DO SUL,AMAZONIA COMERCIO DE DERIVADOS DE PETROLEO LTDA,01.492.748/0003-83,AVENIDA COPACABANA,440,COPACABANA,69980-000,DIESEL,2026-01-02,8.15,R$ / litro,IPIRANGA
2,N,AC,CRUZEIRO DO SUL,AMAZONIA COMERCIO DE DERIVADOS DE PETROLEO LTDA,01.492.748/0003-83,AVENIDA COPACABANA,440,COPACABANA,69980-000,DIESEL S10,2026-01-02,8.17,R$ / litro,IPIRANGA
3,N,AC,RIO BRANCO,AUTO POSTO AMAPA LTDA,00.529.581/0001-53,VIA CHICO MENDES,3570,AREAL,69906-119,GASOLINA,2026-01-02,7.29,R$ / litro,VIBRA
4,N,AC,RIO BRANCO,AUTO POSTO AMAPA LTDA,00.529.581/0001-53,VIA CHICO MENDES,3570,AREAL,69906-119,GASOLINA ADITIVADA,2026-01-02,7.39,R$ / litro,VIBRA
